In [4]:
from pathlib import Path
import pandas as pd

# Resolve path whether notebook is run from project root or notebooks/
candidates = [
    Path("data/raw/global_ads_performance_dataset.csv"),
    Path("../data/raw/global_ads_performance_dataset.csv"),
]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find CSV. Tried: " + ", ".join(str(p) for p in candidates)
    )

df = pd.read_csv(csv_path)

print("Loaded from:", csv_path.resolve())
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

df.head()

Loaded from: /Users/Carys/Documents/Github/ads-analysis/data/raw/global_ads_performance_dataset.csv
Shape: (1800, 14)

Data types:
date              object
platform          object
campaign_type     object
industry          object
country           object
impressions        int64
clicks             int64
CTR              float64
CPC              float64
ad_spend         float64
conversions        int64
CPA              float64
revenue          float64
ROAS             float64
dtype: object


,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39


In [5]:
# Check naming inconsistencies in text columns (whitespace/case variants)
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    s = df[col].dropna().astype(str)

    # 1) Values with leading/trailing spaces
    spaced = s[s != s.str.strip()]
    if not spaced.empty:
        print(f"\n[{col}] values with leading/trailing spaces:")
        print(spaced.value_counts().head(10))

    # 2) Variants that collapse to same normalized form
    normalized = s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    tmp = pd.DataFrame({"raw": s, "norm": normalized})

    counts = (
        tmp.groupby("norm")["raw"]
        .nunique()
        .reset_index(name="n_variants")
        .query("n_variants > 1")
        .sort_values("n_variants", ascending=False)
    )

    if not counts.empty:
        print(f"\n[{col}] possible naming inconsistencies:")
        for norm_key in counts["norm"].head(10):
            variants = tmp.loc[tmp["norm"] == norm_key, "raw"].value_counts()
            print(f"- normalized='{norm_key}' -> {list(variants.index)}")

print("\nDone. Review flagged columns above.")


Done. Review flagged columns above.


In [6]:
# Clean text columns to make naming uniform
# Strategy: trim spaces, collapse internal whitespace, then map variants to one canonical value

text_cols = [c for c in df.select_dtypes(include="object").columns if c != "date"]
changes_summary = []

df_clean = df.copy()

for col in text_cols:
    original = df_clean[col]

    # Basic normalization for matching
    stripped = original.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    normalized = stripped.str.lower()

    # Canonical value per normalized key = most frequent stripped variant
    canon_map = (
        pd.DataFrame({"norm": normalized, "raw": stripped})
        .groupby(["norm", "raw"]).size()
        .reset_index(name="n")
        .sort_values(["norm", "n"], ascending=[True, False])
        .drop_duplicates("norm")
        .set_index("norm")["raw"]
        .to_dict()
    )

    cleaned = normalized.map(canon_map)
    df_clean[col] = cleaned

    n_changed = (original.astype(str) != cleaned.astype(str)).sum()
    if n_changed > 0:
        changes_summary.append((col, int(n_changed)))

# Replace working dataframe
df = df_clean

if changes_summary:
    print("Columns cleaned (values changed):")
    for col, n in changes_summary:
        print(f"- {col}: {n}")
else:
    print("No naming inconsistencies found to clean.")

# Optional spot check
df[text_cols].head()

No naming inconsistencies found to clean.


,platform,campaign_type,industry,country
0,Google Ads,Search,Fintech,UAE
1,TikTok Ads,Search,EdTech,UK
2,TikTok Ads,Video,Healthcare,USA
3,TikTok Ads,Shopping,SaaS,Germany
4,TikTok Ads,Search,Healthcare,UK


In [7]:
# Save processed dataset
processed_candidates = [
    Path("data/processed"),
    Path("../data/processed"),
]

processed_dir = next((p for p in processed_candidates if p.parent.exists()), processed_candidates[0])
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "global_ads_performance_dataset_processed.csv"
df.to_csv(output_path, index=False)

print("Saved processed CSV to:", output_path.resolve())
print("Rows, columns:", df.shape)

Saved processed CSV to: /Users/Carys/Documents/Github/ads-analysis/data/processed/global_ads_performance_dataset_processed.csv
Rows, columns: (1800, 14)
